# KMTNet ToO Pipeline — Step-by-Step Tutorial

Welcome! This notebook walks through the KMTNet
Target-of-Opportunity (ToO) reduction pipeline, **one stage at a time**, on a
small example dataset.

The goal is to *understand* and *test* each stage in isolation. Every stage is
a single function call from the `steps` helper module (see `tutorial/steps.py`),
which hides all the file-name regex and path plumbing.

### The 10 stages
| # | Stage | What it does |
|---|-------|--------------|
| 1 | `ampcom`      | Combine the 32 raw amplifier extensions into 4 chip images (kk/mm/tt/nn) |
| 2 | `astrom`      | Solve the WCS of each chip (SExtractor + SCAMP, Gaia-XP) |
| 3 | `astromqa`    | Validate astrometry + build cosmic-ray / bleed / weight masks |
| 4 | `zpscale`     | Scale each chip to a common photometric zero-point |
| 5 | `bpmask`      | Merge all defect maps into one bad-pixel mask |
| 6 | `stacking`    | Co-add dithers into per-field science + mask stacks (SWarp) |
| 7 | `stackqa`     | Re-run QA on the stacked images |
| 8 | `catalog`     | Build a calibrated source catalogue from each stack |
| 9 | `subtraction` | Difference imaging (HOTPANTS) → **transient candidates** |
| 10| `rbclass`     | Real/Bogus ML classification of the candidate cutouts (optional) |

> **Tip:** Run the cells top to bottom. Each stage feeds the next, so the order
> matters. You can re-run any single stage as often as you like.

## 0. Prerequisites

Before running this notebook, make sure:

1. **Conda environment** — use the `kmtnet` environment, which has the Python
   dependencies (astropy, astroquery, numpy, scipy, matplotlib, astroscrappy, …).
   Launch Jupyter from that environment, or select the `kmtnet` kernel:
   ```bash
   conda activate kmtnet
   jupyter notebook    # or: jupyter lab
   ```
2. **External astronomy software** must be on your `PATH`:
   `source-extractor` (SExtractor), `scamp`, `swarp`, `hotpants`, and optionally
   `psfex`.
3. **Reference data** is already wired up in this repo:
   - Gaia-XP catalogue → `catalog/gaiaxp/`
   - Reference templates → `data/tmpl/` (symlink to the KS4 stacks)
   - SExtractor/SCAMP/SWarp configs → `config/`

You do **not** need to create any directories by hand — `steps` does that for you.

## 1. Import the helper and see the stages

`steps.py` lives next to this notebook. Importing it also makes the core
pipeline importable and creates the base directory tree.

In [ ]:
import os, sys

# Make `steps.py` importable whether Jupyter started in this folder or the repo root.
for _p in (os.getcwd(), os.path.join(os.getcwd(), "tutorial")):
    if os.path.isfile(os.path.join(_p, "steps.py")):
        sys.path.insert(0, _p)
        break

import steps

steps.list_steps()

## 2. Pick a small test dataset

The full `example` directory has ~35 raw frames covering several fields; a
complete run takes ~50 minutes. For learning, we build a **2-frame subset** of
field **0292** (which has a reference template), so the whole chain — including
subtraction — finishes in a few minutes.

`make_subset` just symlinks the chosen raw frames into a new run directory under
`data/raw/`. The "run name" (a directory name) is the single argument every
stage takes.

In [ ]:
# Build the subset and use it as our run name for the rest of the notebook.
# (To process the full example set instead, just set: DATE = "example")
DATE = steps.make_subset(dst_date="quicktest0292", src_date="example",
                         serials=("062875", "062876"))
print("Working on run:", DATE)

## Stage 1 — `ampcom` (amplifier → chip)

A raw KMTNet file has **32 amplifier extensions**. `ampcom` stitches them into
**4 chip images** per frame (`NNNNNN.kk/mm/tt/nn.fits`), does a per-amp sky
subtraction, and quarantines obviously bad frames into `bad*` sub-folders.

After this runs, look in `data/raw/quicktest0292/` — you should see the new
`*.kk/mm/tt/nn.fits` files appear next to the raw symlinks.

In [ ]:
steps.run_ampcom(DATE)
steps.list_outputs(DATE)

## Stage 2 — `astrom` (astrometric calibration)

Runs SExtractor to detect stars on each chip, then SCAMP to solve the WCS
against the local **Gaia-XP** catalogue (offline; UCAC-4 over the network is the
fallback). The TPV solution is written into each chip header. If SCAMP fails to
converge, it automatically retries with a cached "last-good" initial guess for
the same site+chip.

In [ ]:
steps.run_astrom(DATE)

# Peek at the WCS that was written into one chip header.
import glob
from astropy.io import fits
chip = sorted(glob.glob(os.path.join(steps.path_raw, DATE, "??????.??.fits")))[0]
hdr = fits.getheader(chip)
print(os.path.basename(chip))
for k in ("CTYPE1", "CTYPE2", "CRVAL1", "CRVAL2"):
    print(f"  {k} = {hdr.get(k)}")

## Stage 3 — `astromqa` (chip QA + masks)

For each chip this:
- validates the astrometric solution using an **edge-focused** check (only the
  outermost ring of grid sections, where distortion is largest, decides pass/fail),
- builds the **cosmic-ray**, **saturation-bleed**, and **weight** masks that the
  later stages depend on.

A failure on any single chip is caught and reported — it never aborts the run.

In [ ]:
steps.run_astromqa(DATE)

# QA writes ALNRMS (alignment RMS, arcsec) and QARESULT into each chip header.
hdr = fits.getheader(chip)
print(f"{os.path.basename(chip)}:  QARESULT={hdr.get('QARESULT')}  ALNRMS={hdr.get('ALNRMS')}")

## Stage 4 — `zpscale` (photometric scaling)

Every chip is scaled to a common zero-point (default **30 mag**) so frames from
different nights/sites are directly comparable. Outputs land in
`data/scaled/<run>/` with descriptive names like
`FIELD.RADEC.BAND.DATE.SITE.SERIAL.CHIP.scaled.fits`.

In [ ]:
steps.run_zpscale(DATE)

# Quick-look at one scaled chip.
scaled = sorted(glob.glob(os.path.join(steps.path_scale, DATE, "*.scaled.fits")))
print(f"{len(scaled)} scaled chips")
if scaled:
    steps.show_fits(scaled[0])

## Stage 5 — `bpmask` (bad-pixel map)

Combines the cosmic-ray mask, the static bad-pixel map, and bad-amplifier
information into a single mask per scaled chip, using distinct flag values for
each defect type.

In [ ]:
steps.run_bpmask(DATE)

## Stage 6 — `stacking` (co-addition)

Complete `(kk, mm, tt, nn)` dither sets are grouped by KMTNet grid field and
co-added with SWarp, re-projected onto the field's reference frame. This
produces a deep science **stack** and its **mask** in `data/stack/<run>/`.

In [ ]:
steps.run_stacking(DATE)

stacks = sorted(glob.glob(os.path.join(steps.path_stack, DATE, "*.stack.fits")))
print(f"{len(stacks)} field stack(s):", [os.path.basename(s) for s in stacks])
if stacks:
    steps.show_fits(stacks[0])

## Stage 7 — `stackqa` (stack QA)

The same QA logic as stage 3, now applied to the deeper stacked images
(without the per-chip cosmic-ray/bleed masking, which only applies to single
exposures). Writes the alignment metrics into the stack headers.

In [ ]:
steps.run_stackqa(DATE)

## Stage 8 — `catalog` (source catalogue)

Calibrates the SExtractor measurements from each stack against the reference
catalogue, computes the 5σ limiting magnitude, and writes a zero-point-corrected
source catalogue alongside the stack.

In [ ]:
steps.run_catalog(DATE)

## Stage 9 — `subtraction` (difference imaging → transients)

The heart of the pipeline. For each field stack it:
- subtracts the reference template using **HOTPANTS** (over a 4×4 grid of subregions),
- detects sources on the difference image,
- applies the multi-flag artifact filter,
- writes the **transient catalogue** (`*.transient.cat`) and **cutout images**
  into `data/subt/<run>/snap/`.

You can optionally pass a `known_obj` CSV (resolved under `catalog/`) so that
detections matching known targets are always snapshotted regardless of flags.

In [ ]:
# This is the longest stage (HOTPANTS over 16 subregions). A few minutes for the subset.
steps.run_subtraction(DATE)                 # or: steps.run_subtraction(DATE, known_obj="S250206dm/S250206dm.csv")
steps.list_outputs(DATE)

### Inspect the transient candidates

Read the transient catalogue and visualise one candidate's cutout triplet
(new / reference / difference) from the `snap/` folder.

In [ ]:
from astropy.table import Table

subt_dir = os.path.join(steps.path_subt, DATE)
cats = sorted(glob.glob(os.path.join(subt_dir, "*.transient.cat")))
if cats:
    tcat = Table.read(cats[0], format="ascii")
    print(f"{len(tcat)} candidate(s) in {os.path.basename(cats[0])}")
    tcat[:10].pprint(max_width=-1)
else:
    print("No transient catalogue found (the subset may yield zero candidates).")

# Show one cutout if any exist.
snaps = sorted(glob.glob(os.path.join(subt_dir, "snap", "*.fits")))
print(f"\n{len(snaps)} cutout file(s) in snap/")
if snaps:
    steps.show_fits(snaps[0])

## Stage 10 — `rbclass` (Real/Bogus classification, optional)

Runs the KMTNet-specific machine-learning model over the candidate cutouts to
score them as real vs. bogus. This stage is **optional**: it needs the
`rbclass_kmtnet/` model package, and is skipped cleanly if there are no cutouts
or the model isn't installed.

In [ ]:
steps.run_rbclass(DATE)

## Running everything at once

Once you understand the stages, you can run the whole chain with one call:

```python
steps.run_all(DATE)                       # stages 1–10
steps.run_all(DATE, start_from="stacking")  # resume from a given stage
```

…or from a terminal (no notebook needed):

```bash
conda activate kmtnet
python tutorial/steps.py quicktest0292                 # all stages
python tutorial/steps.py quicktest0292 --step ampcom   # one stage
python tutorial/steps.py quicktest0292 --from zpscale  # resume
```

### Cleaning up the test run
The subset and all its products are throwaway. To start fresh, delete the run
directories (they're under the git-ignored `data/`):

```python
import shutil
for sub in (steps.path_raw, steps.path_scale, steps.path_stack, steps.path_subt):
    shutil.rmtree(os.path.join(sub, DATE), ignore_errors=True)
```

That's the whole pipeline — congratulations! For the scientific details of each
stage, see the `README.md` in the repository root and the docstrings in
`pipe/KMTNet_ToO_functions.py`.